OpenAI → 지원서 요약 50명 생성
OpenAI → 각 지원자별 체크리스트 정답 true/false 생성
report.py → 각 지원자별 체크리스트 예측 true/false 생성
비교 → acc, f1, matrix, 문항별 표 출력

In [49]:
from pathlib import Path
import sys
import json
import time
from typing import List

import pandas as pd
from pydantic import BaseModel, Field

# =========================
# 1. 설정
# =========================

PROJECT_ROOT = Path(r"C:\project_skn\final\Final_project")
BACKEND_DIR = PROJECT_ROOT / "backend"
EVAL_DIR = BACKEND_DIR / "common" / "eval"
EVAL_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(BACKEND_DIR))

from common.report import check_resume_fit, _get_openai_client

client = _get_openai_client()

GEN_MODEL = "gpt-4.1-mini"
GOLD_MODEL = "gpt-4.1-mini"

# =========================
# 2. 고정 체크리스트
# =========================

CHECKLIST = [
    "컴퓨터 공학 또는 유사 전공자로서 대졸 이상 학력을 보유하고 있는가?",
    "프론트엔드 관련 경력 3년 이상을 보유하고 있는가?",
    "HTML, CSS, JavaScript 기술 역량을 충분히 갖추고 있는가?",
    "React, Vue, Vite 중 하나 이상의 프레임워크 또는 빌드 도구 사용 경험이 있는가?",
    "SaaS 기반 플랫폼 개발 경험 또는 업무 자동화 서비스 이해도가 있는가?",
    "AI 기반 데이터 분석 기능에 대한 기초 이해와 관심을 가지고 있는가?",
    "팀 내 협업 능력이 뛰어나며 커뮤니케이션 역량이 우수한가?",
    "업무에 대한 책임감을 가지고 자율적으로 과제를 수행한 경험이 있는가?",
    "Python, Java, Node와 같은 백엔드 언어 또는 REST API, DB 설계, 인증/권한에 관한 기본 지식이 있는가?",
    "정규직 근무 형태에 적합하며 장기 근속 의지와 성장 가능성을 갖추었는가?",
]

# =========================
# 3. 지원서 요약 50명 생성
# =========================

class ResumeItem(BaseModel):
    applicant_id: int
    name: str
    resume_summary: str

class ResumeBatch(BaseModel):
    resumes: List[ResumeItem]


RESUME_GENERATION_SYSTEM_PROMPT = """
너는 채용 평가용 가상 지원서 요약 데이터를 만드는 전문가다.
프론트엔드 개발자 채용 평가에 사용할 지원서 요약을 생성한다.

생성 규칙:
- 각 지원자는 서로 다른 배경, 학력, 경력, 기술 스택, 협업 경험, 성장 의지를 가져야 한다.
- 지원서 요약은 사용자가 준 예시처럼 자연스러운 한국어 문단으로 작성한다.
- 체크리스트 10개 항목이 모두 항상 true가 되면 안 된다.
- 어떤 지원자는 프론트엔드 3년 이상이고, 어떤 지원자는 3년 미만이어야 한다.
- 어떤 지원자는 컴퓨터공학 또는 유사 전공 대졸 이상이고, 어떤 지원자는 아니어야 한다.
- 어떤 지원자는 React/Vue/Vite 경험이 있고, 어떤 지원자는 없어야 한다.
- 어떤 지원자는 SaaS, AI 데이터 분석, 백엔드/API/DB/인증 지식이 있고, 어떤 지원자는 없어야 한다.
- true/false가 다양하게 섞이도록 만든다.
- 지원서 요약 안에 노골적으로 "체크리스트 1번 true" 같은 표현은 절대 쓰지 않는다.
- applicant_id는 요청받은 시작 번호부터 순서대로 부여한다.
"""


def generate_resume_batch(start_id, count):
    user_prompt = f"""
지원자 {count}명의 지원서 요약을 생성해줘.

조건:
- applicant_id는 {start_id}부터 {start_id + count - 1}까지 사용
- 각 resume_summary는 6~9문장 정도의 자연스러운 한국어 문단
- 프론트엔드 개발자 채용 평가에 적합한 요약
- 아래 체크리스트 기준에서 true/false가 다양하게 나오도록 지원자 프로필을 섞어줘

고정 체크리스트:
{json.dumps(CHECKLIST, ensure_ascii=False, indent=2)}
"""

    response = client.beta.chat.completions.parse(
        model=GEN_MODEL,
        messages=[
            {"role": "system", "content": RESUME_GENERATION_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=ResumeBatch,
    )

    return [item.model_dump() for item in response.choices[0].message.parsed.resumes]


resume_rows = []

BATCH_SIZE = 10
TOTAL_APPLICANTS = 50

for start_id in range(1, TOTAL_APPLICANTS + 1, BATCH_SIZE):
    print(f"지원서 요약 생성 중: {start_id}~{start_id + BATCH_SIZE - 1}")
    resume_rows.extend(generate_resume_batch(start_id, BATCH_SIZE))
    time.sleep(0.3)

resume_df = pd.DataFrame(resume_rows).sort_values("applicant_id").reset_index(drop=True)

assert len(resume_df) == 50, f"지원서 요약은 50개여야 합니다. 현재 {len(resume_df)}개입니다."

resume_df.to_csv(EVAL_DIR / "generated_resume_summaries_50.csv", index=False, encoding="utf-8-sig")

# =========================
# 4. OpenAI 정답 생성
# =========================

class GoldItem(BaseModel):
    criterion_id: int = Field(description="체크리스트 번호. 1부터 10까지")
    content: str = Field(description="체크리스트 원문")
    expected: bool = Field(description="지원서 요약 기준 정답 true/false")

class GoldResult(BaseModel):
    checklist: List[GoldItem]


GOLD_SYSTEM_PROMPT = """
너는 채용 평가 정답 라벨러다.
지원서 요약과 고정 체크리스트를 보고 각 항목이 충족되는지 true/false로 판단한다.

판단 규칙:
- 지원서 요약에 명확한 근거가 있으면 true.
- 명확한 근거가 없으면 false.
- 애매한 추론, 가능성, 관심만으로는 true로 판단하지 않는다.
- 경력 기간, 학력, 기술명처럼 조건이 있는 항목은 조건을 엄격히 확인한다.
- 부분 충족은 체크리스트 전체 조건을 만족하지 못하면 false다.
- 반드시 체크리스트 1번부터 10번까지 순서대로 판단한다.
- content는 입력 체크리스트 원문을 그대로 사용한다.
"""


def make_gold_labels(resume_summary):
    context = {
        "resume_summary": resume_summary,
        "checklist": [
            {"criterion_id": i, "content": item}
            for i, item in enumerate(CHECKLIST, start=1)
        ],
    }

    response = client.beta.chat.completions.parse(
        model=GOLD_MODEL,
        messages=[
            {"role": "system", "content": GOLD_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    "다음 지원서 요약을 기준으로 체크리스트 정답 expected를 생성해줘.\n\n"
                    + json.dumps(context, ensure_ascii=False, indent=2)
                ),
            },
        ],
        response_format=GoldResult,
    )

    return [item.model_dump() for item in response.choices[0].message.parsed.checklist]


# =========================
# 5. 정답 생성 + report.py 예측
# =========================

gold_rows = []
pred_rows = []
errors = []

for idx, row in resume_df.iterrows():
    applicant_id = row["applicant_id"]
    name = row["name"]
    resume_summary = row["resume_summary"]

    print(f"{idx + 1}/50 평가 중: {name}")

    try:
        gold_items = make_gold_labels(resume_summary)

        for item in gold_items:
            gold_rows.append({
                "applicant_id": applicant_id,
                "name": name,
                "criterion_id": item["criterion_id"],
                "checklist_content": item["content"],
                "expected": item["expected"],
            })

        pred_items = check_resume_fit(
            resume_summary=resume_summary,
            checklist=CHECKLIST,
        )

        for criterion_id, pred in enumerate(pred_items, start=1):
            pred_rows.append({
                "applicant_id": applicant_id,
                "name": name,
                "criterion_id": criterion_id,
                "predicted": pred.get("result"),
            })

    except Exception as e:
        errors.append({
            "applicant_id": applicant_id,
            "name": name,
            "error": repr(e),
        })

    time.sleep(0.2)

gold_df = pd.DataFrame(gold_rows)
pred_df = pd.DataFrame(pred_rows)
error_df = pd.DataFrame(errors)

gold_df.to_csv(EVAL_DIR / "openai_gold_labels_50x10.csv", index=False, encoding="utf-8-sig")
pred_df.to_csv(EVAL_DIR / "report_predictions_50x10.csv", index=False, encoding="utf-8-sig")
error_df.to_csv(EVAL_DIR / "eval_errors.csv", index=False, encoding="utf-8-sig")

if len(error_df):
    print("오류 발생")
    display(error_df)

# =========================
# 6. 정답 vs 예측 비교
# =========================

eval_df = gold_df.merge(
    pred_df,
    on=["applicant_id", "name", "criterion_id"],
    how="left",
)

eval_df["predicted_bool"] = eval_df["predicted"].map(
    lambda x: x if isinstance(x, bool) else None
)

eval_df["is_valid"] = eval_df["predicted_bool"].notna()
eval_df["is_correct"] = eval_df["expected"] == eval_df["predicted_bool"]

valid_df = eval_df[eval_df["is_valid"]].copy()

# =========================
# 7. 전체 지표
# =========================

tp = ((valid_df["expected"] == True) & (valid_df["predicted_bool"] == True)).sum()
tn = ((valid_df["expected"] == False) & (valid_df["predicted_bool"] == False)).sum()
fp = ((valid_df["expected"] == False) & (valid_df["predicted_bool"] == True)).sum()
fn = ((valid_df["expected"] == True) & (valid_df["predicted_bool"] == False)).sum()

accuracy = valid_df["is_correct"].mean() if len(valid_df) else 0
precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0
specificity = tn / (tn + fp) if (tn + fp) else 0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
balanced_accuracy = (recall + specificity) / 2 if len(valid_df) else 0

metrics_df = pd.DataFrame([{
    "total": len(eval_df),
    "valid": len(valid_df),
    "invalid": len(eval_df) - len(valid_df),
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "specificity": specificity,
    "f1_score": f1_score,
    "balanced_accuracy": balanced_accuracy,
    "tp": tp,
    "tn": tn,
    "fp": fp,
    "fn": fn,
}])

display(metrics_df)

# =========================
# 8. Confusion Matrix
# =========================

confusion_matrix = pd.crosstab(
    valid_df["expected"],
    valid_df["predicted_bool"],
    rownames=["expected"],
    colnames=["predicted"],
    dropna=False,
)

confusion_matrix = confusion_matrix.reindex(
    index=[False, True],
    columns=[False, True],
    fill_value=0,
)

display(confusion_matrix)

# =========================
# 9. 체크리스트 문항별 성능표
# =========================

item_metrics_df = (
    valid_df
    .groupby(["criterion_id", "checklist_content"])
    .agg(
        count=("is_correct", "size"),
        accuracy=("is_correct", "mean"),
        expected_true_rate=("expected", "mean"),
        predicted_true_rate=("predicted_bool", "mean"),
    )
    .reset_index()
    .sort_values("criterion_id")
)

display(item_metrics_df)

# =========================
# 10. 저장
# =========================

metrics_df.to_csv(EVAL_DIR / "metrics_summary.csv", index=False, encoding="utf-8-sig")
confusion_matrix.to_csv(EVAL_DIR / "confusion_matrix.csv", encoding="utf-8-sig")
item_metrics_df.to_csv(EVAL_DIR / "item_metrics.csv", index=False, encoding="utf-8-sig")
eval_df.to_csv(EVAL_DIR / "eval_detail.csv", index=False, encoding="utf-8-sig")

print("저장 완료:", EVAL_DIR)

지원서 요약 생성 중: 1~10
지원서 요약 생성 중: 11~20
지원서 요약 생성 중: 21~30
지원서 요약 생성 중: 31~40
지원서 요약 생성 중: 41~50
1/50 평가 중: 이준호
2/50 평가 중: 김서연
3/50 평가 중: 박민재
4/50 평가 중: 최유진
5/50 평가 중: 홍성민
6/50 평가 중: 배지현
7/50 평가 중: 임태현
8/50 평가 중: 강미소
9/50 평가 중: 오현석
10/50 평가 중: 조아라
11/50 평가 중: 김도현
12/50 평가 중: 이지훈
13/50 평가 중: 최세영
14/50 평가 중: 박민서
15/50 평가 중: 한지훈
16/50 평가 중: 이수빈
17/50 평가 중: 정현우
18/50 평가 중: 김예린
19/50 평가 중: 서동혁
20/50 평가 중: 윤채원
21/50 평가 중: 김지훈
22/50 평가 중: 이수민
23/50 평가 중: 박성호
24/50 평가 중: 최예진
25/50 평가 중: 장민석
26/50 평가 중: 한지우
27/50 평가 중: 서다은
28/50 평가 중: 윤태현
29/50 평가 중: 김예린
30/50 평가 중: 황동현
31/50 평가 중: 이성준
32/50 평가 중: 박지우
33/50 평가 중: 최민호
34/50 평가 중: 김하영
35/50 평가 중: 이규민
36/50 평가 중: 정다은
37/50 평가 중: 한승완
38/50 평가 중: 배수진
39/50 평가 중: 송태영
40/50 평가 중: 오지은
41/50 평가 중: 김지훈
42/50 평가 중: 이수연
43/50 평가 중: 박성민
44/50 평가 중: 최민지
45/50 평가 중: 한동훈
46/50 평가 중: 윤서현
47/50 평가 중: 조민재
48/50 평가 중: 서지은
49/50 평가 중: 박지호
50/50 평가 중: 강유진


,total,valid,invalid,accuracy,precision,recall,specificity,f1_score,balanced_accuracy,tp,tn,fp,fn
0,500,500,0,0.958,0.952381,0.984615,0.908571,0.96823,0.946593,320,159,16,5


predicted,False,True
expected,,
False,159,16
True,5,320


,criterion_id,checklist_content,count,accuracy,expected_true_rate,predicted_true_rate
0,1,컴퓨터 공학 또는 유사 전공자로서 대졸 이상 학력을 보유하고 있는가?,50,0.90,0.54,0.56
1,2,프론트엔드 관련 경력 3년 이상을 보유하고 있는가?,50,1.00,0.48,0.48
2,3,"HTML, CSS, JavaScript 기술 역량을 충분히 갖추고 있는가?",50,0.96,0.76,0.80
3,4,"React, Vue, Vite 중 하나 이상의 프레임워크 또는 빌드 도구 사용 경험...",50,1.00,0.60,0.60
4,5,SaaS 기반 플랫폼 개발 경험 또는 업무 자동화 서비스 이해도가 있는가?,50,0.94,0.42,0.44
5,6,AI 기반 데이터 분석 기능에 대한 기초 이해와 관심을 가지고 있는가?,50,0.94,0.42,0.48
6,7,팀 내 협업 능력이 뛰어나며 커뮤니케이션 역량이 우수한가?,50,1.00,0.96,0.96
7,8,업무에 대한 책임감을 가지고 자율적으로 과제를 수행한 경험이 있는가?,50,0.98,0.98,1.00
8,9,"Python, Java, Node와 같은 백엔드 언어 또는 REST API, DB ...",50,0.90,0.42,0.48
9,10,정규직 근무 형태에 적합하며 장기 근속 의지와 성장 가능성을 갖추었는가?,50,0.96,0.92,0.92


저장 완료: C:\project_skn\final\Final_project\backend\common\eval
